In [2]:
pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 12.4 MB/s eta 0:00:00


In [7]:
import io
import os
import json
import hashlib
from typing import Dict, List, Iterable, Optional

import boto3
import pandas as pd
from tqdm import tqdm
from botocore.config import Config
from botocore.exceptions import ClientError
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

try:
    from google.colab import userdata
except ImportError:
    userdata = None


# =========================
# CONFIG
# =========================
AWS_ACCESS_KEY_ID = userdata.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = userdata.get("AWS_SECRET_ACCESS_KEY")
AWS_REGION = userdata.get("AWS_DEFAULT_REGION") or "us-east-2"

# Cleaned source bucket + CUAD root
SOURCE_BUCKET = "cleaned-legal-data-analysis-src-v2"

# For max speed, keep only the prefix that truly contains the final cleaned section files.
# Based on your screenshots, sections/ is the best first choice.
SOURCE_PREFIXES = [
    "CUAD_v1/sections/",
    "CUAD_v1/cleaned_sections/",
]

# Shared target vector store
TARGET_VECTOR_BUCKET = "combined.embeddings"
TARGET_INDEX_NAME = "legal-combined-index"

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
INDEX_DIMENSION = 384
DISTANCE_METRIC = "cosine"
DATA_TYPE = "float32"

MAX_TOKENS = 350
OVERLAP_TOKENS = 75

# Bigger = faster, until Colab RAM/GPU becomes a problem
EMBED_BATCH_CHUNKS = 256
PUT_BATCH_SIZE = 25
GET_EXISTING_BATCH_SIZE = 100

# Keep only section-like files
ALLOWED_SUFFIXES = (".parquet", ".jsonl", ".json")


# =========================
# CLIENTS
# =========================
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION,
)

cfg = Config(retries={"max_attempts": 10, "mode": "standard"})
s3 = session.client("s3", config=cfg)
s3vectors = session.client("s3vectors", config=cfg)


# =========================
# HELPERS
# =========================
def list_s3_keys(bucket: str, prefix: str) -> Iterable[str]:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue
            lower = key.lower()
            if lower.endswith(ALLOWED_SUFFIXES):
                yield key


def read_s3_object(bucket: str, key: str) -> bytes:
    return s3.get_object(Bucket=bucket, Key=key)["Body"].read()


def read_dataframe_from_s3(bucket: str, key: str) -> Optional[pd.DataFrame]:
    raw = read_s3_object(bucket, key)
    lower = key.lower()

    if lower.endswith(".parquet"):
        return pd.read_parquet(io.BytesIO(raw))

    if lower.endswith(".jsonl"):
        rows = []
        for line in raw.decode("utf-8", errors="ignore").splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                continue
        return pd.DataFrame(rows) if rows else None

    if lower.endswith(".json"):
        text = raw.decode("utf-8", errors="ignore").strip()
        if not text:
            return None
        try:
            obj = json.loads(text)
        except Exception:
            return None

        if isinstance(obj, list):
            return pd.DataFrame(obj)
        if isinstance(obj, dict):
            if "rows" in obj and isinstance(obj["rows"], list):
                return pd.DataFrame(obj["rows"])
            if "data" in obj and isinstance(obj["data"], list):
                return pd.DataFrame(obj["data"])
            return pd.DataFrame([obj])

    return None


def clean_text(text) -> str:
    if text is None:
        return ""
    text = str(text).replace("\x00", " ")
    text = " ".join(text.split())
    return text.strip()


def pick_first(row: Dict, candidates: List[str], default: str = "") -> str:
    for c in candidates:
        if c in row:
            val = row[c]
            if pd.notna(val):
                val = clean_text(val)
                if val:
                    return val
    return default


def normalize_cuad_row(row: Dict, source_uri: str, row_idx: int) -> Optional[Dict]:
    text = pick_first(
        row,
        [
            "section_text",
            "text",
            "clean_text",
            "cleaned_text",
            "content",
            "body",
            "clause_text",
            "section_content",
        ],
    )
    if not text:
        return None

    doc_id = pick_first(
        row,
        [
            "doc_id",
            "document_id",
            "contract_id",
            "agreement_id",
            "file_id",
            "file_name",
            "filename",
            "id",
        ],
        default=f"cuad_doc_{row_idx}",
    )

    section_title = pick_first(
        row,
        [
            "section_title",
            "section",
            "heading",
            "title",
            "header",
            "clause_type",
            "label",
        ],
        default="",
    )

    return {
        "dataset": "cuad",
        "doc_id": doc_id,
        "section_title": section_title,
        "text": text,
        "source_uri": source_uri,
        "row_idx": row_idx,
    }


def chunk_text(text: str, tokenizer, max_tokens: int, overlap_tokens: int) -> List[str]:
    text = clean_text(text)
    if not text:
        return []

    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
    )["input_ids"]

    if not token_ids:
        return []

    if len(token_ids) <= max_tokens:
        chunk = tokenizer.decode(token_ids, skip_special_tokens=True).strip()
        return [chunk] if chunk else []

    chunks = []
    step = max_tokens - overlap_tokens
    for start in range(0, len(token_ids), step):
        end = start + max_tokens
        piece = token_ids[start:end]
        if not piece:
            continue
        chunk = tokenizer.decode(piece, skip_special_tokens=True).strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(token_ids):
            break
    return chunks


def make_vector_key(
    dataset: str,
    doc_id: str,
    source_uri: str,
    row_idx: int,
    chunk_idx: int,
    chunk_text_value: str,
) -> str:
    base = f"{dataset}|{doc_id}|{source_uri}|row={row_idx}|chunk={chunk_idx}"
    base_hash = hashlib.md5(base.encode("utf-8")).hexdigest()[:20]
    txt_hash = hashlib.md5(chunk_text_value.encode("utf-8")).hexdigest()[:12]
    return f"{dataset}::{base_hash}::{txt_hash}"


def chunked(lst: List, n: int) -> Iterable[List]:
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


def ensure_vector_bucket_and_index():
    bucket_exists = True
    try:
        s3vectors.get_vector_bucket(vectorBucketName=TARGET_VECTOR_BUCKET)
    except ClientError as e:
        if e.response["Error"]["Code"] in {"NotFoundException", "ResourceNotFoundException"}:
            bucket_exists = False
        else:
            raise

    if not bucket_exists:
        s3vectors.create_vector_bucket(vectorBucketName=TARGET_VECTOR_BUCKET)

    index_exists = True
    try:
        s3vectors.get_index(
            vectorBucketName=TARGET_VECTOR_BUCKET,
            indexName=TARGET_INDEX_NAME,
        )
    except ClientError as e:
        if e.response["Error"]["Code"] in {"NotFoundException", "ResourceNotFoundException"}:
            index_exists = False
        else:
            raise

    if not index_exists:
        s3vectors.create_index(
            vectorBucketName=TARGET_VECTOR_BUCKET,
            indexName=TARGET_INDEX_NAME,
            dataType=DATA_TYPE,
            dimension=INDEX_DIMENSION,
            distanceMetric=DISTANCE_METRIC,
        )


def existing_keys(keys: List[str]) -> set:
    found = set()
    for batch in chunked(keys, GET_EXISTING_BATCH_SIZE):
        try:
            resp = s3vectors.get_vectors(
                vectorBucketName=TARGET_VECTOR_BUCKET,
                indexName=TARGET_INDEX_NAME,
                keys=batch,
            )
            for v in resp.get("vectors", []):
                k = v.get("key")
                if k:
                    found.add(k)
        except ClientError:
            pass
    return found


def put_vectors(vectors: List[Dict]):
    for batch in chunked(vectors, PUT_BATCH_SIZE):
        s3vectors.put_vectors(
            vectorBucketName=TARGET_VECTOR_BUCKET,
            indexName=TARGET_INDEX_NAME,
            vectors=batch,
        )


def flush_pending(model, pending_rows: List[Dict], stats: Dict[str, int]) -> List[Dict]:
    if not pending_rows:
        return []

    unique_rows = {}
    for r in pending_rows:
        unique_rows[r["key"]] = r
    pending_rows = list(unique_rows.values())

    keys = [r["key"] for r in pending_rows]
    already = existing_keys(keys)

    to_insert = [r for r in pending_rows if r["key"] not in already]
    stats["skipped_existing"] += len(pending_rows) - len(to_insert)
    stats["attempted_chunks"] += len(pending_rows)

    if not to_insert:
        return []

    texts = [r["chunk_text"] for r in to_insert]
    embeddings = model.encode(
        texts,
        normalize_embeddings=True,
        batch_size=128,
        show_progress_bar=False,
    )

    payload = []
    seen_payload_keys = set()

    for row, emb in zip(to_insert, embeddings):
        if row["key"] in seen_payload_keys:
            continue
        seen_payload_keys.add(row["key"])

        payload.append(
            {
                "key": row["key"],
                "data": {"float32": [float(x) for x in emb]},
                "metadata": {
                    "dataset": "cuad",
                    "doc_id": row["doc_id"][:500],
                    "section_title": row["section_title"][:500],
                    "source_uri": row["source_uri"][:1000],
                },
            }
        )

    if payload:
        put_vectors(payload)
        stats["inserted_chunks"] += len(payload)

    return []


# =========================
# MAIN
# =========================
def main():
    print("Loading embedding model...")
    model = SentenceTransformer(EMBED_MODEL_NAME)
    tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)

    print("Ensuring target vector bucket and index exist...")
    ensure_vector_bucket_and_index()

    print("Discovering CUAD files...")
    keys = []
    for prefix in SOURCE_PREFIXES:
        keys.extend(list(list_s3_keys(SOURCE_BUCKET, prefix)))

    seen = set()
    filtered_keys = []
    for k in keys:
        if k not in seen:
            seen.add(k)
            filtered_keys.append(k)

    print(f"Files found: {len(filtered_keys)}")

    pending_rows = []
    stats = {
        "files_seen": 0,
        "rows_seen": 0,
        "normalized_rows": 0,
        "attempted_chunks": 0,
        "inserted_chunks": 0,
        "skipped_existing": 0,
    }

    for key in tqdm(filtered_keys, desc="Processing CUAD files"):
        stats["files_seen"] += 1

        df = read_dataframe_from_s3(SOURCE_BUCKET, key)
        if df is None or df.empty:
            continue

        source_uri = f"s3://{SOURCE_BUCKET}/{key}"
        rows = df.fillna("").to_dict(orient="records")

        for i, row in enumerate(rows):
            stats["rows_seen"] += 1

            norm = normalize_cuad_row(row, source_uri, i)
            if not norm:
                continue

            stats["normalized_rows"] += 1
            chunks = chunk_text(norm["text"], tokenizer, MAX_TOKENS, OVERLAP_TOKENS)
            if not chunks:
                continue

            for chunk_idx, chunk_text_value in enumerate(chunks):
                vector_key = make_vector_key(
                    dataset="cuad",
                    doc_id=norm["doc_id"],
                    source_uri=norm["source_uri"],
                    row_idx=norm["row_idx"],
                    chunk_idx=chunk_idx,
                    chunk_text_value=chunk_text_value,
                )
                pending_rows.append(
                    {
                        "key": vector_key,
                        "doc_id": norm["doc_id"],
                        "section_title": norm["section_title"],
                        "source_uri": norm["source_uri"],
                        "chunk_text": chunk_text_value,
                    }
                )

            if len(pending_rows) >= EMBED_BATCH_CHUNKS:
                pending_rows = flush_pending(model, pending_rows, stats)
                print(
                    f"Inserted={stats['inserted_chunks']}  "
                    f"Skipped={stats['skipped_existing']}  "
                    f"Attempted={stats['attempted_chunks']}"
                )

    if pending_rows:
        pending_rows = flush_pending(model, pending_rows, stats)

    print("\nDone.")
    print(f"Files seen        : {stats['files_seen']}")
    print(f"Rows seen         : {stats['rows_seen']}")
    print(f"Normalized rows   : {stats['normalized_rows']}")
    print(f"Attempted chunks  : {stats['attempted_chunks']}")
    print(f"Inserted chunks   : {stats['inserted_chunks']}")
    print(f"Skipped existing  : {stats['skipped_existing']}")


if __name__ == "__main__":
    main()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ensuring target vector bucket and index exist...
Discovering CUAD files...
Files found: 5


Processing CUAD files:   0%|          | 0/5 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 512). Running this sequence through the model will result in indexing errors


...

Processing CUAD files:  80%|████████  | 4/5 [46:14<11:39, 699.41s/it]

Inserted=94767  Skipped=0  Attempted=94767
Inserted=95083  Skipped=0  Attempted=95083
Inserted=95414  Skipped=0  Attempted=95414
Inserted=95692  Skipped=0  Attempted=95692
Inserted=95971  Skipped=0  Attempted=95971
Inserted=96255  Skipped=0  Attempted=96255
Inserted=96579  Skipped=0  Attempted=96579
Inserted=96837  Skipped=0  Attempted=96837
Inserted=97144  Skipped=0  Attempted=97144
Inserted=97533  Skipped=0  Attempted=97533
Inserted=97799  Skipped=0  Attempted=97799
Inserted=98056  Skipped=0  Attempted=98056
Inserted=98386  Skipped=0  Attempted=98386
Inserted=98925  Skipped=0  Attempted=98925
Inserted=99193  Skipped=0  Attempted=99193
Inserted=99476  Skipped=0  Attempted=99476
Inserted=99756  Skipped=0  Attempted=99756
Inserted=100039  Skipped=0  Attempted=100039
Inserted=100363  Skipped=0  Attempted=100363
Inserted=100670  Skipped=0  Attempted=100670
Inserted=100961  Skipped=0  Attempted=100961
Inserted=101242  Skipped=0  Attempted=101242
Inserted=101501  Skipped=0  Attempted=101501

Processing CUAD files: 100%|██████████| 5/5 [56:07<00:00, 673.49s/it]

Inserted=114746  Skipped=0  Attempted=114746



Done.
Files seen        : 5
Rows seen         : 36006
Normalized rows   : 36006
Attempted chunks  : 114787
Inserted chunks   : 114787
Skipped existing  : 0
